In [1]:
import json
import hashlib
import sys
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "phi4-mini:3.8b-q4_K_M"
API_LOCAL = "http://127.0.0.1:11434"

# Leer archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
configuracion = json.loads(bytes_config)

# Comprobar integridad
sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == configuracion["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert configuracion["api_url"] == API_LOCAL + "/api/chat"

# Conectar únicamente al servidor local
cliente_local = build_opener(ProxyHandler({}))

with cliente_local.open(API_LOCAL + "/api/version", timeout=10) as r:
    version_ollama = json.load(r)["version"]

with cliente_local.open(API_LOCAL + "/api/tags", timeout=10) as r:
    modelos = json.load(r)["models"]

modelo_info = next(
    (m for m in modelos if m["name"] == MODELO),
    None
)

assert modelo_info is not None, "Phi-4-mini no está instalado."

# Consultar características y capacidades
peticion = Request(
    API_LOCAL + "/api/show",
    data=json.dumps({"model": MODELO}).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

with cliente_local.open(peticion, timeout=15) as r:
    detalles = json.load(r)

print("Python:", sys.version.split()[0])
print("Ollama:", version_ollama)
print("Modelo:", MODELO)
print("Identificador:", modelo_info["digest"][:12])
print("Casos verificados:", len(casos))
print("SHA256:", sha_casos)
print("Prompt:", configuracion["prompt_version"])
print("Capacidades:", detalles.get("capabilities"))
print("Thinking:", detalles.get("thinking"))
print("Parámetros:", detalles.get("parameters"))
print("\nArchivos y conexión: correctos")


Python: 3.11.16
Ollama: 0.40.13
Modelo: phi4-mini:3.8b-q4_K_M
Identificador: 78fad5d182a7
Casos verificados: 6
SHA256: f73d159bb43cc9c93d2768e997052f4381a2c24bb046c82826b1793c1b8da37d
Prompt: V3
Capacidades: ['completion', 'tools']
Thinking: None
Parámetros: None

Archivos y conexión: correctos


In [2]:
import json
import copy
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request

# Caso diagnóstico usado con los modelos anteriores
contenido = (
    "Autores: Autor X y Autor Y.\n"
    "Predicción recibida: favorable (simulada).\n"
    "Modelo predictor y puntuación: no disponibles.\n"
    "[E1] Autor X tiene un artículo sobre bases de datos académicas.\n"
    "[E2] Autor Y tiene un artículo sobre recuperación de información.\n"
    "No se proporcionan datos de colaboración previa."
)

# Reutilizar la configuración compartida
solicitud = copy.deepcopy(
    configuracion["request_defaults"]
)

solicitud["model"] = MODELO

# Phi-4-mini no anuncia un modo thinking configurable
solicitud.pop("think", None)

solicitud["options"].update({
    "num_ctx": 4096,
    "num_predict": 2048,
    "temperature": 0.0
})

# Mantener el prompt V3
solicitud["messages"] = [
    {
        "role": "system",
        "content": configuracion["system_prompt"]
    },
    {
        "role": "user",
        "content": configuracion["user_prompt_template"].format(
            case_id="DIAG-P4M-01",
            contenido=contenido
        )
    }
]

# Verificar que no exista un diagnóstico anterior
try:
    with open("resultados.jsonl", encoding="utf-8") as f:
        anteriores = [
            json.loads(line) for line in f if line.strip()
        ]
except FileNotFoundError:
    anteriores = []

assert not any(
    r["case_id"] == "DIAG-P4M-01" for r in anteriores
), "El diagnóstico ya está registrado."

# Preparar el registro
registro = {
    "execution_id": str(uuid4()),
    "case_id": "DIAG-P4M-01",
    "grupo": "diagnostico",
    "configuracion_modelo": "phi4mini_directa_v1",
    "modelo_digest": modelo_info["digest"],
    "ollama_version": version_ollama,
    "sha256_casos": sha_casos,
    "sha256_configuracion": sha_config,
    "fecha_inicio_utc": datetime.now(timezone.utc).isoformat(),
    "solicitud": solicitud,
    "reintentos": 0
}

inicio = time.perf_counter()

try:
    peticion = Request(
        configuracion["api_url"],
        data=json.dumps(
            solicitud, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente_local.open(peticion, timeout=240) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"
    elif respuesta.get("message", {}).get("thinking"):
        registro["estado"] = "revisar_modo"
    elif (
        respuesta.get("done_reason") == "stop"
        and respuesta.get("message", {}).get("content", "").strip()
    ):
        registro["estado"] = "ok"
    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error_tipo"] = type(error).__name__
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)
registro["fecha_fin_utc"] = datetime.now(timezone.utc).isoformat()

# Guardar el diagnóstico, incluso si falla
with open("resultados.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(registro, ensure_ascii=False) + "\n")

print("Estado:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]

    print("Tokens entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))
    print("Razonamiento separado:", bool(
        r.get("message", {}).get("thinking")
    ))

    for campo in (
        "total_duration",
        "load_duration",
        "prompt_eval_duration",
        "eval_duration"
    ):
        valor = r.get(campo)
        if valor is not None:
            print(campo, round(valor / 1e9, 3), "s")

    print("\nRESPUESTA FINAL:\n")
    print(r.get("message", {}).get("content", ""))
else:
    print("Error:", registro.get("error"))

print("\nRegistro guardado en resultados.jsonl")


Estado: ok
Tiempo cliente: 14.136 s
Tokens entrada: 340
Tokens generados: 212
Terminación: stop
Razonamiento separado: False
total_duration 14.134 s
load_duration 2.356 s
prompt_eval_duration 9.556 s
eval_duration 2.212 s

RESPUESTA FINAL:

Evidencia: 
- [E1] Autor X tiene un artículo sobre bases de datos académicas.
- [E2] Autor Y tiene un artículo sobre recuperación de información.

Interpretación: 
La predicción favorable se basa en la similitud temática entre los artículos de Autor X y Autor Y. Ambas áreas de estudio, bases de datos académicas y recuperación de información, están relacionadas con la gestión y el acceso a datos en entornos académicos. Esta similitud temática sugiere que ambos autores podrían tener intereses o necesidades académicas similares, lo que podría llevar a una colaboración futura.

Limitaciones: 
La evidencia proporcionada no demuestra una colaboración previa entre los autores. La predicción se basa únicamente en la similitud temática de sus trabajos, sin e

In [3]:
import json
import copy
import hashlib
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

ARCHIVO = "resultados.jsonl"
MODELO = "phi4-mini:3.8b-q4_K_M"

# 1. Verificar los archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
config = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == config["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

# 2. Recuperar el diagnóstico
with open(ARCHIVO, encoding="utf-8") as f:
    anteriores = [
        json.loads(line)
        for line in f if line.strip()
    ]

diagnostico = next(
    r for r in anteriores
    if r["case_id"] == "DIAG-P4M-01"
)

assert diagnostico["estado"] == "ok"
assert diagnostico["solicitud"]["model"] == MODELO
assert diagnostico["solicitud"]["options"]["temperature"] == 0.0

# 3. Verificar la instalación local
cliente = build_opener(ProxyHandler({}))

with cliente.open(
    "http://127.0.0.1:11434/api/tags",
    timeout=10
) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    m for m in modelos if m["name"] == MODELO
)

assert modelo_info["digest"] == diagnostico["modelo_digest"]

# Evitar ejecuciones duplicadas
registrados = {
    r["case_id"] for r in anteriores
    if r.get("configuracion_modelo") == "phi4mini_directa_v1"
    and r.get("grupo") == "comparacion"
}

# 4. Ejecutar los casos
with open(ARCHIVO, "a", encoding="utf-8") as archivo:

    for caso in casos:

        case_id = caso["case_id"]

        if case_id in registrados:
            print(case_id, "| ya registrado")
            continue

        # Reutilizar la solicitud del diagnóstico
        solicitud = copy.deepcopy(
            diagnostico["solicitud"]
        )

        assert solicitud["messages"][0]["content"] == (
            config["system_prompt"]
        )

        solicitud["messages"][1]["content"] = (
            config["user_prompt_template"].format(
                case_id=case_id,
                contenido=caso["contenido"]
            )
        )

        registro = {
            "execution_id": str(uuid4()),
            "case_id": case_id,
            "tipo": caso["tipo"],
            "grupo": "comparacion",
            "configuracion_modelo": "phi4mini_directa_v1",
            "prompt_version": "V3",
            "modelo_digest": modelo_info["digest"],
            "ollama_version": diagnostico["ollama_version"],
            "sha256_casos": sha_casos,
            "sha256_configuracion": sha_config,
            "fecha_inicio_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "solicitud": solicitud,
            "reintentos": 0
        }

        inicio = time.perf_counter()

        try:
            peticion = Request(
                config["api_url"],
                data=json.dumps(
                    solicitud, ensure_ascii=False
                ).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST"
            )

            with cliente.open(peticion, timeout=240) as conexion:
                respuesta = json.load(conexion)

            registro["respuesta_original"] = respuesta

            mensaje = respuesta.get("message", {})

            if respuesta.get("done_reason") == "length":
                registro["estado"] = "truncada"

            elif mensaje.get("thinking"):
                registro["estado"] = "revisar_modo"

            elif (
                respuesta.get("done_reason") == "stop"
                and mensaje.get("content", "").strip()
            ):
                registro["estado"] = "ok"

            else:
                registro["estado"] = "revisar"

        except Exception as error:
            registro["estado"] = "error"
            registro["error_tipo"] = type(error).__name__
            registro["error"] = str(error)

        registro["tiempo_cliente_s"] = round(
            time.perf_counter() - inicio, 3
        )

        registro["fecha_fin_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        # Guardar cada ejecución inmediatamente
        archivo.write(
            json.dumps(registro, ensure_ascii=False) + "\n"
        )
        archivo.flush()

        r = registro.get("respuesta_original", {})

        print(
            case_id,
            "|", registro["estado"],
            "|", r.get("eval_count", "-"), "tokens",
            "|", registro["tiempo_cliente_s"], "s"
        )

print("\nArchivo actualizado:", ARCHIVO)


S-005 | ok | 202 tokens | 4.169 s
S-006 | ok | 207 tokens | 1.797 s
S-007 | ok | 207 tokens | 1.931 s
S-008 | ok | 205 tokens | 2.113 s
S-009 | ok | 268 tokens | 2.95 s
S-010 | ok | 142 tokens | 1.513 s

Archivo actualizado: resultados.jsonl
